In [19]:
import pandas as pd
from tqdm.std import tqdm
from openai import OpenAI
import re
import os
import json

In [20]:
input_data = "example_input_files/semantic_matching_0099.csv"
output_data = "example_output_files/semantic_matching_0099.xlsx"

In [36]:
## Open AI API
os.environ["OPENAI_API_KEY"] = "your open ai api"

## common functions

In [22]:
def to_pascal_case(phrase):
    # Remove parentheses and the content inside them, e.g., (Unaudited)
    phrase = re.sub(r'\s*\([^)]*\)', '', phrase)
    
    # Split on non-alphanumeric characters
    words = re.split(r'[^a-zA-Z0-9]', phrase)
    
    # Capitalize each word and join
    return ''.join(word.capitalize() for word in words if word)

In [23]:
def read_xml(xml_path):
    with open(xml_path, "r", encoding="utf-8") as f:
        content = f.read()

        return content

In [35]:
"""
    This function is different a bit among different tasks, here is for the semantic matching task
"""

def construct_annotation(df, results, task_name="semantic_matching"):
    """
        the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
    """
    new_df = df.copy(deep=True)

    schema = []
    presentation = []
    calculation = []
    definition = []
    label = []
    instance = []
    us_gaap = []
    true_answer = []
    segmentation_path = []
    parse_ok = []
    input = []
    outputByLLM = []
    
    for res in results:
        segmentation_name = res.get("segmentation_name")
        folder_name = res.get("folder_name")

        prefix_path = os.path.join(f"second_round_filter/segmentation_filing/{task_name}", folder_name)
        real_segmentation_names = os.listdir(prefix_path)
        potential_seg_names = []

        ## obtain the potential segmentation folders using partial match
        for seg_name in real_segmentation_names:
            normalized_segmentation_name = segmentation_name.lower()
            normalized_seg_name = seg_name.lower()
            if (normalized_segmentation_name in normalized_seg_name) or (normalized_seg_name in normalized_segmentation_name):
                potential_seg_names.append(seg_name)
        
        parse = False
        cal = None
        pre = None
        defn = None
        lab = None
        ins = None
        sch = None
        seg_path = None
        answers = [tag.get("tag") for tag in res.get("error_tags")]
        for segmentation_name_ in potential_seg_names:
            
            full_path = os.path.join(prefix_path, segmentation_name_)
    
            temp_cal = None
            temp_pre = None
            temp_defn = None
            temp_lab = None
            temp_ins = None
            temp_sch = None
            
            file_list = os.listdir(full_path)
            
            for file_name in file_list:
                if file_name.endswith("cal.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_cal = read_xml(final_path)
                elif file_name.endswith("pre.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_pre = read_xml(final_path)
                elif file_name.endswith("def.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_defn = read_xml(final_path)
                elif file_name.endswith("lab.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_lab = read_xml(final_path)
                elif file_name.endswith("htm.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_ins = read_xml(final_path)
                elif file_name.endswith("xsd"):
                    final_path = os.path.join(full_path,file_name)
                    temp_sch = read_xml(final_path)
                else:
                    continue

            is_appear = True
            for ans in answers:
                # check the presentation linkbase
                if ans in temp_pre:
                    continue
                elif ans.replace(":","") in temp_pre:
                    continue
                elif ans.replace(":","_") in temp_pre:
                    continue
                else:
                    is_appear = False
                    break
           
            if is_appear:
                parse = True
                cal = temp_cal
                pre = temp_pre
                defn = temp_defn
                lab = temp_lab
                ins = temp_ins
                sch = temp_sch
                seg_path = os.path.join(folder_name, segmentation_name_)
                break
        
        calculation.append(cal)
        presentation.append(pre)
        definition.append(defn)
        label.append(lab)
        instance.append(ins)
        schema.append(sch)
        true_answer.append(answers)
        us_gaap.append(None)
        parse_ok.append(parse)
        segmentation_path.append(seg_path)
        input.append(None)
        outputByLLM.append(res)
     
        
    new_df["schema"] = schema
    new_df["presentation"] = presentation
    new_df["calculation"] = calculation
    new_df["definition"] = definition
    new_df["label"] = label
    new_df["instance"] = instance
    new_df["us_gaap"] = us_gaap
    new_df["input"] = input
    new_df["true_answer"] = true_answer
    new_df["segmentation_path"] = segmentation_path
    new_df["parse_successful"] = parse_ok
    new_df["LLM_Output"] = outputByLLM

    return new_df

## load dqc messages for semantic matching task

In [25]:
sm_99_df = pd.read_csv(input_data)

In [26]:
sm_99_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,report.base-taxonomy,report.creation-software,report.document-type,report.entry-url,report.filing-date,report.filing-year,report.id,report.sec-url,report.sic-code,folder_name
0,DQC.US.0099.9533,"2024-03-26 11:19:44,533 [DQC.US.0099.9533] \nT...",2024-03-26,ERROR,DQC,99,1319643,"Lode-Star Mining, Inc.",2024-03-22 15:06:00,0001199835-24-000120,US GAAP 2023,Novaworks,10-K,http://www.sec.gov/Archives/edgar/data/1319643...,2024-03-22,2024,692753,https://www.sec.gov/Archives/edgar/data/131964...,1040,10k-lsmg-20231231
1,DQC.US.0099.9533,"2022-02-15 07:30:54,506 [DQC.US.0099.9533] \nT...",2022-02-15,ERROR,DQC,99,1620749,PANAMERA HOLDINGS CORPORATION,2022-02-14 21:51:00,0001640334-22-000315,US GAAP 2021,Document Created by XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/1620749...,2022-02-15,2022,419692,https://www.sec.gov/Archives/edgar/data/162074...,8742,10k-pnht-20210731
2,DQC.US.0099.9533,"2022-11-17 06:28:43,249 [DQC.US.0099.9533] \nT...",2022-11-17,ERROR,DQC,99,1876945,Gold Flora Corporation,2022-11-14 16:56:00,0001654954-22-015300,US GAAP 2022,Document Created by XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1876945...,2022-11-14,2022,529878,https://www.sec.gov/Archives/edgar/data/187694...,100,10q-gramf-20220930
3,DQC.US.0099.9533,"2023-12-12 22:04:07,575 [DQC.US.0099.9533] \nT...",2023-12-13,ERROR,DQC,99,1616741,"PATRIOT\nTRANSPORTATION HOLDING, INC.",2023-12-12 15:49:00,0001616741-23-000042,US GAAP 2023,Novaworks,10-K,http://www.sec.gov/Archives/edgar/data/1616741...,2023-12-12,2023,661477,https://www.sec.gov/Archives/edgar/data/161674...,4210,10k-pati-20230930
4,DQC.US.0099.9533,"2022-02-18 09:45:49,004 [DQC.US.0099.9533] \nT...",2022-02-18,ERROR,DQC,99,1602813,FRONTERA GROUP INC.,2022-02-18 09:24:00,0001549727-22-000009,US GAAP 2020,Field: Doc-Info; Name: Generator; Value: GoFil...,10-Q,http://www.sec.gov/Archives/edgar/data/1602813...,2022-02-18,2022,422094,https://www.sec.gov/Archives/edgar/data/160281...,8742,10q-frtg-20211231
5,DQC.US.0099.9533,"2022-01-18 07:01:54,632 [DQC.US.0099.9533] \nT...",2022-01-18,ERROR,DQC,99,1796949,"GenFlat Holdings, Inc.",2022-01-18 06:02:00,0001654954-22-000549,US GAAP 2020,Document Created by XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1796949...,2022-01-18,2022,412314,https://www.sec.gov/Archives/edgar/data/179694...,8742,10q-hbr-20211130
6,DQC.US.0099.9533,"2024-08-14 06:08:49,558 [DQC.US.0099.9533] \nT...",2024-08-14,ERROR,DQC,99,1411342,ELLINGTON FINANCIAL INC,2024-08-09 15:35:00,0001411342-24-000078,US GAAP 2023,XBRL Document Created with the Workiva Platform,10-Q,http://www.sec.gov/Archives/edgar/data/1411342...,2024-08-09,2024,749558,https://www.sec.gov/Archives/edgar/data/141134...,6500,10q-efc-20240630
7,DQC.US.0099.9533,"2024-11-15 10:17:04,302 [DQC.US.0099.9533] \nT...",2024-11-15,ERROR,DQC,99,1411342,ELLINGTON FINANCIAL INC,2024-11-12 15:47:00,0001411342-24-000113,US GAAP 2023,XBRL Document Created with the Workiva Platform,10-Q,http://www.sec.gov/Archives/edgar/data/1411342...,2024-11-12,2024,779538,https://www.sec.gov/Archives/edgar/data/141134...,6500,10q-efc-20240930
8,DQC.US.0099.9533,"2023-02-02 19:01:53,630 [DQC.US.0099.9533] \nT...",2023-02-03,ERROR,DQC,99,1604778,"Qorvo, Inc.",2023-02-02 16:10:00,0001604778-23-000034,US GAAP 2022,XBRL Document Created with Wdesk from Workiva,10-Q,http://www.sec.gov/Archives/edgar/data/1604778...,2023-02-02,2023,545306,https://www.sec.gov/Archives/edgar/data/160477...,3674,10q-rfmd-20221231
9,DQC.US.0099.9533,"2023-02-23 15:34:42,815 [DQC.US.0099.9533] \nT...",2023-02-23,ERROR,DQC,99,1065696,LKQ CORPORATION,2023-02-23 15:08:00,0001065696-23-000008,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/1065696...,2023-02-23,2023,551270,https://www.sec.gov/Archives/edgar/data/106569...,5010,10k-l

## LLM judge assisstant

In [27]:
Client = OpenAI()

In [28]:
sm_99_system_prompt = """You are given a DQC validation message. Your task is to extract two specific pieces of information:

1. **Statement Name**: Extract the name of the financial statement mentioned in the message. If it appears in a phrase like "0000007 - Statement - Consolidated Statements of Operations", ignore any prefixes such as "- Statement -" or ID numbers. Only return the clean name of the statement (e.g., "Consolidated Statements of Operations").

2. **Erroneous us-gaap Tag(s)**: From all `us-gaap:` prefixed tags mentioned in the message, extract only those that are clearly described as having an error or violation. A tag should be included **only if** it satisfies both of the following:
   - It is explicitly stated to be **not included in any calculation relationship**.
   - The message states that it **will produce an error**, **violates a rule**, or **should have been reported differently**.
Do not include `us-gaap:` tags that are merely suggested, referenced as valid abstract parents, or shown as examples for correction.

3. **Reasoning**: For each extracted erroneous tag, provide a short explanation summarizing why it was classified as erroneous, based on the message.

Output your result as a structured JSON in the following format:
```json
{
  "statement_name": "...",
  "error_tags": [
    {
      "tag": "...",
      "reason": "..."
    }
  ]
}
"""

In [29]:
def get_response(dqc_message):
    
    completion = Client.chat.completions.create(
        model="gpt-4.1-mini",
        response_format={
            "type": "json_object"
        },
        messages=[
            {"role": "system", "content": sm_99_system_prompt},
            {"role": "user", "content": f"DQC validation message: {dqc_message} \n Output:"}
        ]
    )
    return completion.choices[0].message.content

In [30]:
res = []
for _, row in tqdm(sm_99_df.iterrows()):
    ## load each dqc message
    message = row["assertion.detail"]
    response = get_response(message)
    response = json.loads(response)

    ## load the folder name which will be used to obtain candidate segmentation folders
    folder_name = row["folder_name"]

    ## covert the extracted statement name in the message to PascalCase format as the segmentation name
    segmentation_name = to_pascal_case(response.get("statement_name"))

    ## record the folder name
    response["folder_name"] = folder_name
    ## record the extracted segmenataion name
    response["segmentation_name"] = segmentation_name
    res.append(response)

10it [00:48,  4.84s/it]


In [31]:
res[0]

{'statement_name': 'STATEMENTS OF CASH FLOWS',
 'error_tags': [{'tag': 'us-gaap:IncreaseDecreaseInPrepaidExpense',
   'reason': 'The tag is included in the presentation but not in any calculation relationship, which will produce an error.'},
  {'tag': 'us-gaap:IncreaseDecreaseInInterestPayableNet',
   'reason': 'The tag is included in the presentation but not in any calculation relationship, which will produce an error.'},
  {'tag': 'us-gaap:NetIncomeLoss',
   'reason': 'The tag is included in the presentation but not in any calculation relationship, which will produce an error.'},
  {'tag': 'us-gaap:IncreaseDecreaseInDueToRelatedParties',
   'reason': 'The tag is included in the presentation but not in any calculation relationship, which will produce an error.'},
  {'tag': 'us-gaap:IncreaseDecreaseInAccountsPayableAndAccruedLiabilities',
   'reason': 'The tag is included in the presentation but not in any calculation relationship, which will produce an error.'}],
 'folder_name': '10k-

In [32]:
"""
    the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
"""
new_sm_99_df = construct_annotation(sm_99_df, res, task_name="semantic_matching")

In [33]:
new_sm_99_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,...,calculation,definition,label,instance,us_gaap,input,true_answer,segmentation_path,parse_successful,LLM_Output
0,DQC.US.0099.9533,"2024-03-26 11:19:44,533 [DQC.US.0099.9533] \nT...",2024-03-26,ERROR,DQC,99,1319643,"Lode-Star Mining, Inc.",2024-03-22 15:06:00,0001199835-24-000120,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[us-gaap:IncreaseDecreaseInPrepaidExpense, us-...",10k-lsmg-20231231/StatementsOfCashFlows,True,"{'statement_name': 'STATEMENTS OF CASH FLOWS',..."
1,DQC.US.0099.9533,"2022-02-15 07:30:54,506 [DQC.US.0099.9533] \nT...",2022-02-15,ERROR,DQC,99,1620749,PANAMERA HOLDINGS CORPORATION,2022-02-14 21:51:00,0001640334-22-000315,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[us-gaap:LiabilitiesAndStockholdersEquity, us-...",10k-pnht-20210731/BalanceSheets,True,"{'statement_name': 'Balance Sheets', 'error_ta..."
2,DQC.US.0099.9533,"2022-11-17 06:28:43,249 [DQC.US.0099.9533] \nT...",2022-11-17,ERROR,DQC,99,1876945,Gold Flora Corporation,2022-11-14 16:56:00,0001654954-22-015300,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:CommonStockValue],10q-gramf-20220930/InterimCondensedConsolidate...,True,{'statement_name': 'Interim Condensed Consolid...
3,DQC.US.0099.9533,"2023-12-12 22:04:07,575 [DQC.US.0099.9533] \nT...",2023-12-13,ERROR,DQC,99,1616741,"PATRIOT\nTRANSPORTATION HOLDING, INC.",2023-12-12 15:49:00,0001616741-23-000042,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:RightOfUseAssetObtainedInExchangeForO...,10k-pati-20230930/ConsolidatedStatementsOfCash...,True,{'statement_name': 'CONSOLIDATED STATEMENTS OF...
4,DQC.US.0099.9533,"2022-02-18 09:45:49,004 [DQC.US.0099.9533] \nT...",2022-02-18,ERROR,DQC,99,1602813,FRONTERA GROUP INC.,2022-02-18 09:24:00,0001549727-22-000009,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,None,None,None,[us-gaap:IncreaseDecreaseInAccountsPayableAndA...,10q-frtg-20211231/StatementsOfCashFlows,True,{'statement_name': 'STATEMENTS OF CASH FLOWS (...
5,DQC.US.0099.9533,"2022-01-18 07:01:54,632 [DQC.US.0099.9533] \nT...",2022-01-18,ERROR,DQC,99,1796949,"GenFlat Holdings, Inc.",2022-01-18 06:02:00,0001654954-22-000549,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[],10q-hbr-20211130/ConsolidatedStatementsOfOpera...,True,{'statement_name': 'Consolidated Statements of...
6,DQC.US.0099.9533,"2024-08-14 06:08:49,558 [DQC.US.0099.9533] \nT...",2024-08-14,ERROR,DQC,99,1411342,ELLINGTON FINANCIAL INC,2024-08-09 15:35:00,0001411342-24-000078,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:IncreaseDecreaseInMortgageLoansHeldFo...,10q-efc-20240630/ConsolidatedStatementofCashFlows,True,{'statement_name': 'Consolidated Statement of ...
7,DQC.US.0099.9533,"2024-11-15 10:17:04,302 [DQC.US.0099.9533] \nT...",2024-11-15,ER

In [34]:
new_sm_99_df.to_excel(output_data,index=False)